# Sesión 6 · Construcción de tableros con Dash
**Misión 9–11 nov 2026 · Miércoles 11 · 09:00–10:30 · Santiago Quintana**

Notebook de **demostración** (primera mitad, ~45 min). En la segunda mitad: `S6_taller_estudiantes.ipynb`.

Construiremos **por etapas** un tablero **descriptivo** del dataset `penguins` (333 pingüinos de 3 especies y 3 islas):

| Etapa | Min | Qué se agrega |
|---|---|---|
| 0. Usuario → pregunta → KPI | 5 | Qué debe responder el tablero |
| 1. HTML en Dash y `app_v1` | 12 | `html.Div`, `html.H1`, estilos, maqueta con filas y tarjetas |
| 2. `app_v2`: gráficos y tabla | 10 | `dcc.Graph`, Plotly Express, `DataTable` |
| 3. `app_v3`: controles y callback | 13 | Dropdown, Checklist, `Input` / `Output`, los `id` |
| 4. Cómo se corre fuera del notebook | 5 | `app.py`, `python app.py`, navegador |

> Cada app se ve dentro del notebook con `app.run(jupyter_mode="inline")`. En Colab funciona igual. Si no se ve, usa `jupyter_mode="external"` y abre el enlace.

In [1]:
# En Google Colab: instalar lo que no viene por defecto (en local puede omitirse)
import importlib.util, subprocess, sys
for pkg in ["dash", "plotly"]:
    if importlib.util.find_spec(pkg.replace('-', '_')) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

In [2]:
import pandas as pd
import seaborn as sns
import plotly.express as px
from dash import Dash, html, dcc, dash_table, Input, Output

penguins = sns.load_dataset("penguins").dropna().reset_index(drop=True)
ISLAS = sorted(penguins["island"].unique())
ESPECIES = sorted(penguins["species"].unique())
COLORES_ESPECIE = {"Adelie": "#E07B39", "Chinstrap": "#8E44AD", "Gentoo": "#16A085"}
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
4,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male


## 0. Usuario → pregunta → KPI

Antes del código, definimos **para quién** es el tablero y **qué** debe responder.

| Usuario | Pregunta | KPI | Gráfico |
|---|---|---|---|
| Investigador/a | ¿Cuántos pingüinos hay en la selección? | Número de pingüinos | Tabla |
| Investigador/a | ¿Qué tan grandes son? | Masa media (kg) | Histograma de masa por especie |
| Investigador/a | ¿Hay balance entre machos y hembras? | % de machos | Scatter aleta vs masa (símbolo = sexo) |

**KPI** (indicador clave de desempeño): un número que resume si vamos bien en algo que le importa al usuario.

## 1. HTML en Dash y `app_v1`

Una página web es un **árbol de cajas**. En Dash cada etiqueta HTML es una clase de Python:

| Dash | HTML | Para qué |
|---|---|---|
| `html.Div([...])` | `<div>` | caja o contenedor: agrupa otros componentes |
| `html.H1("...")` | `<h1>` | título |
| `html.P("...")` | `<p>` | párrafo de texto |
| `html.Label("...")` | `<label>` | nombre de un control |
| `style={...}` | `style="..."` | color, tamaño, márgenes (CSS en *camelCase*) |

- El **primer argumento** (`children`) es el contenido: un texto o una **lista** de componentes.
- Para poner cajas **una al lado de otra** (una fila): `style={"display": "flex", "gap": "16px"}` en el Div que las contiene; `"flex": "1"` en cada caja reparte el ancho.

In [3]:
# Del código a la página: un Div con un título, un párrafo y una fila de dos cajas
mini = Dash(__name__)
mini.layout = html.Div([
    html.H1("Hola Dash"),
    html.P("Este texto es un html.P dentro de un html.Div."),
    html.Div([
        html.Div("Caja 1", style={"backgroundColor": "#D3E8FF", "padding": "20px", "flex": "1"}),
        html.Div("Caja 2", style={"backgroundColor": "#F7F2ED", "padding": "20px", "flex": "1"}),
    ], style={"display": "flex", "gap": "16px"}),          # fila: las cajas quedan lado a lado
], style={"fontFamily": "Arial", "padding": "20px"})

In [ ]:
mini.run(jupyter_mode="inline", port=8050, height=260)

Para no repetir estilos, los guardamos en diccionarios y creamos una función `tarjeta()` para los KPI:

In [4]:
# Estilos reutilizables (CSS escrito como diccionarios de Python)
COLOR = {"fondo": "#F5F7FA", "texto": "#1F2937", "primario": "#1F4E79", "suave": "#6B7280"}
ESTILO_PAGINA = {"fontFamily": "Arial, sans-serif", "backgroundColor": COLOR["fondo"],
                 "padding": "20px", "color": COLOR["texto"]}
ESTILO_TARJETA = {"backgroundColor": "white", "borderRadius": "10px", "padding": "14px 18px",
                  "boxShadow": "0 1px 4px rgba(0,0,0,.08)", "flex": "1", "minWidth": "160px",
                  "boxSizing": "border-box"}
ESTILO_FILA = {"display": "flex", "gap": "16px", "flexWrap": "wrap", "marginBottom": "16px"}
ESTILO_CAJA = {"backgroundColor": "white", "borderRadius": "10px", "padding": "8px",
               "boxShadow": "0 1px 4px rgba(0,0,0,.08)", "flex": "1", "minWidth": "380px"}

def tarjeta(titulo, valor, id_valor=None):
    """Tarjeta KPI: título pequeño + valor grande. Con id_valor, el valor se puede actualizar con un callback."""
    props = {"id": id_valor} if id_valor else {}
    return html.Div([
        html.Div(titulo, style={"fontSize": "13px", "color": COLOR["suave"]}),
        html.Div(valor, style={"fontSize": "28px", "fontWeight": "700", "color": COLOR["primario"]}, **props),
    ], style=ESTILO_TARJETA)

**Maqueta del tablero** (cada rectángulo es un `html.Div`):

```
html.Div([                     # página        → style=ESTILO_PAGINA
    html.H1("Título"),
    html.Div([...filtros]),    # fila de filtros
    html.Div([tarjeta, tarjeta, tarjeta]),     # fila de KPIs  → style=ESTILO_FILA
    html.Div([dcc.Graph, dcc.Graph]),          # fila de gráficos → style=ESTILO_FILA
    dash_table.DataTable(...),
])
```

In [5]:
def kpis(df):
    if len(df) == 0:   # filtros sin datos: guion en lugar de 'nan'
        return {"n": "0", "masa": "—", "machos": "—"}
    return {"n": f"{len(df)}",
            "masa": f"{df['body_mass_g'].mean() / 1000:.2f} kg",
            "machos": f"{100 * (df['sex'] == 'Male').mean():.0f} %"}

k = kpis(penguins)
app_v1 = Dash(__name__)
app_v1.layout = html.Div([
    html.H1("Estación Palmer · Pingüinos", style={"marginBottom": "4px"}),
    html.P("Tablero descriptivo · seaborn penguins (333 pingüinos)", style={"color": COLOR["suave"]}),
    html.Div([tarjeta("Pingüinos", k["n"]), tarjeta("Masa media", k["masa"]),
              tarjeta("% machos", k["machos"])], style=ESTILO_FILA),
    html.Div([html.Div("Gráfico 1 (próximamente)", style=ESTILO_CAJA),
              html.Div("Gráfico 2 (próximamente)", style=ESTILO_CAJA)], style=ESTILO_FILA),
], style=ESTILO_PAGINA)
k

{'n': '333', 'masa': '4.21 kg', 'machos': '50 %'}

In [ ]:
app_v1.run(jupyter_mode="inline", port=8051, height=380)

## 2. `app_v2` · Gráficos y tabla
Separamos **datos → figuras → layout**: cada función recibe un DataFrame y devuelve una figura. Luego el callback solo tendrá que volver a llamarlas con los datos filtrados.

**Insertar una gráfica en el HTML:** 1) crear la figura con Plotly Express, 2) ponerla en la página con `dcc.Graph(figure=fig)` dentro de un `html.Div`.

In [6]:
def fig_scatter(df):
    fig = px.scatter(df, x="flipper_length_mm", y="body_mass_g", color="species", symbol="sex",
                     color_discrete_map=COLORES_ESPECIE, hover_data=["island", "bill_length_mm"],
                     labels={"flipper_length_mm": "Aleta (mm)", "body_mass_g": "Masa (g)"},
                     title="Aleta y masa crecen juntas")
    return fig.update_layout(margin=dict(t=50, l=10, r=10, b=10))

def fig_hist(df):
    fig = px.histogram(df, x="body_mass_g", color="species", barmode="overlay", opacity=0.7,
                       color_discrete_map=COLORES_ESPECIE, labels={"body_mass_g": "Masa (g)"},
                       title="Distribución de masa por especie")
    return fig.update_layout(margin=dict(t=50, l=10, r=10, b=10))

tabla_cols = ["species", "island", "sex", "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
fig_scatter(penguins)

In [7]:
app_v2 = Dash(__name__)
app_v2.layout = html.Div([
    html.H1("Estación Palmer · Pingüinos"),
    html.Div([tarjeta("Pingüinos", k["n"]), tarjeta("Masa media", k["masa"]),
              tarjeta("% machos", k["machos"])], style=ESTILO_FILA),
    html.Div([html.Div(dcc.Graph(figure=fig_scatter(penguins)), style=ESTILO_CAJA),
              html.Div(dcc.Graph(figure=fig_hist(penguins)), style=ESTILO_CAJA)], style=ESTILO_FILA),
    dash_table.DataTable(data=penguins[tabla_cols].to_dict("records"),
                         columns=[{"name": c, "id": c} for c in tabla_cols],
                         page_size=8, sort_action="native", filter_action="native"),
], style=ESTILO_PAGINA)

In [ ]:
app_v2.run(jupyter_mode="inline", port=8052, height=1000)

## 3. `app_v3` · Controles y callback

Estructura de un **callback**:

```
@app.callback(
    Output("k-n", "children"),        # qué se actualiza: un texto...
    Output("g-scatter", "figure"),    # ...o una figura
    Input("f-isla", "value"),         # qué control ejecuta la función
)
def actualizar(islas):                # recibe el valor de cada Input
    ...
    return texto, figura              # uno por cada Output, en el mismo orden
```
- Los **id** conectan el layout con el callback: deben ser idénticos en ambos lados.
- Flujo: cambia el Dropdown → Dash ejecuta `actualizar` → la figura nueva reemplaza a la anterior.

Buena práctica: la lógica en una función normal (`filtrar`) que se puede probar antes de conectarla.

In [8]:
def filtrar(df, islas, especies):
    return df[df["island"].isin(islas or []) & df["species"].isin(especies or [])]

# Prueba rápida ANTES de conectarla a la app
f = filtrar(penguins, ["Biscoe"], ["Gentoo"])
print(len(f), kpis(f))

119 {'n': '119', 'masa': '5.09 kg', 'machos': '51 %'}


In [9]:
app_v3 = Dash(__name__)
app_v3.layout = html.Div([
    html.H1("Estación Palmer · Pingüinos"),
    html.Div([
        html.Div([html.Label("Isla"),
                  dcc.Dropdown(id="f-isla", options=ISLAS, value=ISLAS, multi=True)], style={"flex": "1"}),
        html.Div([html.Label("Especie"),
                  dcc.Checklist(id="f-especie", options=ESPECIES, value=ESPECIES, inline=True)], style={"flex": "1"}),
    ], style={**ESTILO_TARJETA, "display": "flex", "gap": "20px", "marginBottom": "16px"}),
    html.Div([tarjeta("Pingüinos", "", "k-n"), tarjeta("Masa media", "", "k-masa"),
              tarjeta("% machos", "", "k-machos")], style=ESTILO_FILA),
    html.Div([html.Div(dcc.Graph(id="g-scatter"), style=ESTILO_CAJA),
              html.Div(dcc.Graph(id="g-hist"), style=ESTILO_CAJA)], style=ESTILO_FILA),
    dash_table.DataTable(id="tabla", columns=[{"name": c, "id": c} for c in tabla_cols],
                         page_size=8, sort_action="native", filter_action="native"),
], style=ESTILO_PAGINA)

@app_v3.callback(
    Output("k-n", "children"), Output("k-masa", "children"), Output("k-machos", "children"),
    Output("g-scatter", "figure"), Output("g-hist", "figure"), Output("tabla", "data"),
    Input("f-isla", "value"), Input("f-especie", "value"),
)
def actualizar(islas, especies):
    df = filtrar(penguins, islas, especies)
    k = kpis(df)
    return k["n"], k["masa"], k["machos"], fig_scatter(df), fig_hist(df), df[tabla_cols].to_dict("records")

> **Probar en vivo:** deja solo la isla Torgersen: ¿qué especies quedan? Desmarca todas las especies: ¿se rompe algo?

In [ ]:
app_v3.run(jupyter_mode="inline", port=8053, height=1150)

In [10]:
# Prueba sin servidor, incluyendo un filtro extremo (ninguna especie marcada)
_ = actualizar(ISLAS, ESPECIES)
vacio = actualizar(ISLAS, [])
print("Callback OK · sin especies:", vacio[:3])

Callback OK · sin especies: ('0', '—', '—')


## 4. ¿Cómo se corre una app Dash fuera del notebook?

1. Copia en un archivo `app.py`: los imports, los datos, los estilos, las funciones, el layout y el callback.
2. Al final del archivo agrega:

```python
if __name__ == "__main__":
    app.run(debug=True)
```
3. En la terminal: `python app.py` → abre **http://127.0.0.1:8050** en el navegador. La app queda corriendo hasta `Ctrl + C`.
4. Con `debug=True` la app se recarga sola al guardar cambios y muestra los errores (por ejemplo, un `id` mal escrito) en la página.

**Checklist de diseño:** KPIs arriba y detalle abajo · títulos que dicen el hallazgo · un color = un significado · unidades en ejes y tarjetas.

➡️ **Ahora: `S6_taller_estudiantes.ipynb` (45 min).**